# 06 — Why MCP exists

## How we got here

Module 02: the model does not call the tool. It emits a name and JSON. **Your code** runs `get_fact`.

Modules 03–05: a loop around that. Skills. A coding agent. Still one rule — the function lived **in the notebook kernel**. Same process as the loop.

That is how every first agent is born. It does not travel.

If tomorrow you open LangGraph, or Claude Desktop, or a second kernel, you copy `get_fact` again. Three copies. Three bugs. The loop **owns** the hands.

**MCP (Model Context Protocol)** is the open standard for moving the hands **out** of the loop: **agent → tool**. A small server process owns `get_fact` and `get_flight`. Any host that speaks the protocol can ask "what tools do you have?" and "run this one."

You are not learning a new way for the model to call a function. The model still does not call anything. You are learning a **new address** for the same two tools.

## What you benefit

- Write the travel tools once. Any compatible host can call the same `server.py` instead of copying the functions into its own process.
- A host you did not write (VS Code, Claude Desktop, another language) can use them if it speaks MCP.
- You can prove the tools **without a model** (`tools/list`, `tools/call`). That habit is the point of testing a server first.

MCP is agent-to-tool. A2A is agent-to-agent. Do not mix them. A2A is module 13.

The end of this notebook has a short recap of the parts of the MCP standard we did not use today, for the record.


## 1. Learn

```
02     the model emits a name; your function runs in this kernel
03–05  a loop around those functions — still this kernel
06     you are here — same two travel tools, other process, a protocol
07     async / gather — why every modern host says await
08     the Agents SDK adds sessions and delegation to local Chinook tools
09     LangGraph adds explicit state and control flow to local Chinook tools
```

Three names:

| Name | Role today |
|---|---|
| **Host** | The notebook. Later: VS Code, Claude Desktop, a framework. |
| **Server** | `server.py`. Owns the two tools. |
| **Transport** | **stdio** — we start the server as a subprocess and talk on stdin/stdout. HTTP is the same JSON on a port, for another machine. We stay on stdio. |

### What JSON-RPC is

RPC means **remote procedure call**: run a function that does not live in this process. JSON-RPC is a small, stateless way to do that with JSON. It does not care whether the bytes go over HTTP, a socket, or stdin/stdout.

A request is a JSON object with four ideas:

- `jsonrpc` — always `"2.0"`
- `method` — the name of the procedure
- `params` — the arguments (object or list)
- `id` — so you can match the reply to the ask

Example, from the JSON-RPC idea (not one of our tools):

```
{
  "jsonrpc": "2.0",
  "method": "subtract",
  "params": [42, 23],
  "id": 1
}
```

The other side would return something like `{"jsonrpc": "2.0", "result": 19, "id": 1}`. Same `id`. That is the whole trick.

A **notification** is the same object **without** an `id`. It is a fire-and-forget. We use one after handshake: `notifications/initialized`. We do not wait for a reply.

If you want the spec in a sentence more: [JSON-RPC](https://en.wikipedia.org/wiki/JSON-RPC).

### How MCP uses it

MCP does not invent a new message format. **Every MCP request, response, and notification is a JSON-RPC 2.0 message.** MCP names the methods (`initialize`, `tools/list`, `tools/call`). On stdio the JSON needs a wrapper so the other process knows where one message ends.

Two wrappers you will see:

- **Content-Length.** A header, then the bytes — the same idea VS Code's own autocomplete uses under the hood (the Language Server Protocol, which borrowed this framing from JSON-RPC first). We print this one — you can count them.
- **Newline-delimited JSON.** One object per line. The official Python MCP client, and therefore `MCPServerStdio` in the Agents SDK, speaks this one.

Same JSON object inside. We print the header form once. The official SDK owns the envelope when the file runs as a process.

Our `tools/call` is the `subtract` example with a different method name:

```
{
  "jsonrpc": "2.0",
  "id": 3,
  "method": "tools/call",
  "params": {"name": "get_fact", "arguments": {"city": "Amsterdam"}}
}
```

`handle` is the function that receives that object and either lists tools or runs `get_fact`. The official SDK would do the same routing. We wrote it so you can see it.

**Sandboxing** is not JSON-RPC. It is a host policy: the server process should not get a free pass to the whole disk or network, and a desktop host should ask before a sensitive tool runs. We already refused `../.env` in module 04. Module 10 is a tighter sandbox. Module 14 is what happens when a tool description or a tool result is hostile. Today we only need: the messages are auditable JSON. You can log every ask.

```mermaid
flowchart LR
    A["notebook (host)"] -->|"tools/list"| B["server.py — own process"]
    B -->|"the two schemas"| A
    A -->|"tools/call get_fact"| B
    B -->|"runs get_fact, returns text"| A
    L["the official loop"] -->|"still decides whether to call"| A
```

### Map of `server.py` — two tools, the rest is plumbing

Open `modules/06_mcp/server.py`. Only **two names are tools** (what a host may call).

| In the file | Kind | Job |
|---|---|---|
| `get_fact` | **Tool** | Same function as module 02. Decorated so the official SDK can list it. |
| `get_flight` | **Tool** | Same function as module 02. |
| `handle` | Receptionist | In-process only. `tools/list` and `tools/call` as JSON-RPC, no subprocess. Not a tool. |
| `MCPServer` / `mcp.run("stdio")` | Official SDK | What runs when you start the file. Speaks the wire for any host. |

`handle` exists so you can see `tools/list` and `tools/call` as ordinary Python, with no model and no pipe. A `.py` file is allowed here because the server **must** be its own process. We import `handle` first, then a host we did not write starts the file.


## 2. Do

### Load the environment and import the server as a module

Importing does **not** start the stdio loop. That loop is only under `if __name__ == "__main__"`. So this cell still runs `get_fact` in this kernel — module 02 — as a sanity check that the file loaded.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = os.environ.get("MODEL_DEFAULT", "").strip()
assert api_key, "OPENAI_API_KEY is missing."
assert model, "MODEL_DEFAULT is missing."

client = OpenAI()
HERE = ROOT / "modules" / "06_mcp"
SERVER = HERE / "server.py"

sys.path.insert(0, str(HERE))
import server as travel_mcp

print("OPENAI_API_KEY is set:", True)
print("MODEL_DEFAULT:", model)
print("server file:", SERVER)
print("in-process get_fact:", travel_mcp.get_fact("Amsterdam"))


OPENAI_API_KEY is set: True
MODEL_DEFAULT: gpt-5.4-nano
server file: /Users/tarekatwan/Downloads/ai_agents_course/modules/06_mcp/server.py
in-process get_fact: Amsterdam has more bicycles than people.


That last line is module 02 again: a function in this kernel. The next cells make the **same** function answer from another process.

### `handle` with no subprocess

JSON in, JSON out. No model. Prove the tools work before any host is involved.


In [2]:
listed = travel_mcp.handle(
    {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}
)
print(json.dumps(listed, indent=2)[:700])


{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "tools": [
      {
        "name": "get_fact",
        "description": "Fun fact about a city.",
        "inputSchema": {
          "type": "object",
          "properties": {
            "city": {
              "type": "string"
            }
          },
          "required": [
            "city"
          ]
        }
      },
      {
        "name": "get_flight",
        "description": "Flight price and duration between two cities.",
        "inputSchema": {
          "type": "object",
          "properties": {
            "from_city": {
              "type": "string"
            },
            "to_city": {
              "type": "string"
  


In [3]:
called = travel_mcp.handle(
    {
        "jsonrpc": "2.0",
        "id": 2,
        "method": "tools/call",
        "params": {"name": "get_fact", "arguments": {"city": "Amsterdam"}},
    }
)
print(json.dumps(called, indent=2))


{
  "jsonrpc": "2.0",
  "id": 2,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "Amsterdam has more bicycles than people."
      }
    ]
  }
}


The fact came back inside `result.content[0].text`. That envelope is MCP. The string inside is the same Amsterdam sentence as module 02.

### What goes on the wire

A request is not a bare JSON line. It is a header, a blank line, then the bytes. Print one framed message. We are not sending it yet.


In [4]:
def frame(msg):
    body = json.dumps(msg).encode("utf-8")
    return b"Content-Length: " + str(len(body)).encode("ascii") + b"\r\n\r\n" + body


sample = {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}
raw = frame(sample)
print(raw.decode("utf-8"))
print("bytes:", len(raw))


Content-Length: 65

{"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}
bytes: 87


That is JSON-RPC (`method`, `id`, `params`) plus a `Content-Length` header. The official Agents SDK host later will send the **same JSON** as one line, no header. Cut first, if the room is behind: keep this cell and one `tools/call`.


## 3. Observe

Compare two places the Amsterdam fact can come from, still in this kernel. Then a host you did not write will start the same file.


In [5]:
print("in-process:    ", travel_mcp.get_fact("Amsterdam"))
print("handle() here: ", called["result"]["content"][0]["text"])
print("same string:   ", travel_mcp.get_fact("Amsterdam") == called["result"]["content"][0]["text"])


in-process:     Amsterdam has more bicycles than people.
handle() here:  Amsterdam has more bicycles than people.
same string:    True


One implementation. Two callers so far: a function, and `handle` in this kernel. That is the protocol, with no pipe.

The next cells are the payoff: a host **you did not write** talking to the same `server.py`.


### The SDK as the host

You just saw `tools/list` and `tools/call` as JSON. That is the host, by hand, in-process.

The benefit of MCP is that **someone else's host** can do that. The OpenAI Agents SDK is one such host.

`MCPServerStdio` starts `server.py` for you, lists the tools, and turns `tool_calls` into `tools/call`. You do not frame bytes. This host speaks **one JSON line per message**, not the `Content-Length` header you printed. Same tools. The official `MCPServer` in `server.py` owns the envelope.

The two tools are still `get_fact` and `get_flight`. The SDK does not invent them. This cell uses `await Runner.run`. Jupyter allows it. Module 07 is *why* that word exists. If the cell errors, skip it; `handle` already proved the server.

`with trace("06 MCP Istanbul")` records MCP tool spans, not just Python function tools. Open the printed URL. `draw_graph` should show the agent and a grey box for the MCP server. The grey box *is* `server.py`.


In [6]:
from agents import Agent, Runner, trace, gen_trace_id
from agents.mcp import MCPServerStdio

mcp_server = MCPServerStdio(
    name="travel",
    params={
        "command": sys.executable,
        "args": [str(SERVER)],
        "cwd": str(ROOT),
    },
    client_session_timeout_seconds=30,
)

trace_id = gen_trace_id()
print("Trace:", "https://platform.openai.com/traces/trace?trace_id=" + trace_id)

async with mcp_server:
    mcp_agent = Agent(
        name="Travel via MCP",
        instructions="Use the MCP tools. Do not invent prices or facts.",
        model=model,
        mcp_servers=[mcp_server],
    )
    with trace("06 MCP Istanbul", trace_id=trace_id):
        mcp_result = await Runner.run(
            mcp_agent,
            "Give me a fun fact about Istanbul.",
        )
print(mcp_result.final_output)


Trace: https://platform.openai.com/traces/trace?trace_id=trace_c90adbfe847b423b828c2801467255aa


HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Istanbul is the only city in the world that spans two continents: Europe and Asia.


### Now look at the trace

Open the printed URL (or [platform.openai.com/traces](https://platform.openai.com/traces)). You should see MCP tool spans — the host listed `get_fact` from the other process and called it.

Now the map of the agent plus the server:


In [ ]:
try:
    from agents.extensions.visualization import draw_graph
    from IPython.display import display

    display(draw_graph(mcp_agent))
except Exception as exc:
    print("Graph not drawn:", exc)
    print("Use the trace link. MCP servers show up as a grey box when graphviz is installed.")


The grey box (if the graph drew) is `server.py`. The SDK listed its tools and called them. That is the portability claim: **one server, a host you did not write**.

Module 08 will stay on this SDK and add sessions and delegation. Module 09 will use local Chinook tools so the framework comparison isolates orchestration and state.


### What we didn't show you

Everything above was the real thing, not a simplified stand-in: real JSON-RPC 2.0 messages, the real envelope shapes, the real SDK running `server.py` as its own process for you. A few parts of the MCP standard, named here for completeness, we did not touch today:

| Part of the standard | What we did instead |
|---|---|
| Streamable HTTP — the transport for a server on another machine | We stayed on stdio: one machine, one subprocess. |
| Resources and prompts — a server can offer more than tools | This server only exposes tools. |
| The MCP Inspector — a browser tool for poking at a server by hand | We poked at ours with `handle()` instead, no browser needed. |

Everything else in that comparison — the JSON-RPC envelope, the methods, the tool shape, the SDK on stdio — is exactly what you ran above, not a simplification of it.

## 4. Challenge

Use `handle` in-process. No model required:

- `tools/list` — how many tools?
- `tools/call` `get_flight` from Sydney to Madrid

Bind:

- `n_tools` — length of the list the server returned
- `flight` — the text of the flight result

The check does not print the price. We will look at it in the debrief.


In [ ]:
# n_tools, flight = ...


In [ ]:
assert n_tools >= 1, "tools/list should come back with at least one tool"
assert flight and "dollar" in flight.lower(), "flight should be the text from tools/call"
print("n_tools:", n_tools)
print(flight)
